# TOPOTEX Model Inspector — Orientation-Aligned Single-Image Prototype

当前唯一被检查的模型：**Canonical Mesh + 1 张 stochastic rendered image →
SingleImageEncoder + FaceTokenizer → Face-Image Cross Attention → Z_F →
Factorized UV Query → Flow Matching → texture**。所有张量来自公开 API 的
真实 forward；不复制模型代码。

In [ ]:
import json, os, sys, types
from pathlib import Path
p = Path.cwd()
PROJECT_ROOT = next(c for c in (p, *p.parents) if (c / "configs").exists())
sys.path.insert(0, str(PROJECT_ROOT))

OA_ROOT = Path("/root/youjiaZhang/topotex_data_OA")
RUN_DIR = Path(os.environ.get("INSPECT_RUN", OA_ROOT / "runs/oa80_smoke_single"))
SAMPLE_ID = os.environ.get("INSPECT_SAMPLE") or None
RANDOM_SEED = None          # None = 每次执行随机换 case；设整数可复现
DEVICE = "cuda:0"

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "Noto Sans CJK JP", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

from topotex import TopoTexDataset, TopoTexPipeline

def resolve_ckpt(run_dir):
    for name in ("ckpt.pt", "ckpt_final.pt"):
        if (Path(run_dir) / name).exists():
            return Path(run_dir) / name
    raise FileNotFoundError(f"no checkpoint under {run_dir}")

pipe = TopoTexPipeline.from_checkpoint(resolve_ckpt(RUN_DIR), DEVICE)
ck, model = pipe.checkpoint, pipe.model
assert model.conditioner.image_encoder_kind == "single", "OA inspector expects the single-image branch"
man = [json.loads(l) for l in open(OA_ROOT / "dataset/manifest.jsonl")]
cat_of = {m["sample_id"]: m.get("lvis_category") for m in man}
rng = np.random.default_rng(RANDOM_SEED)
SEED = int(rng.integers(2**31))
print("run:", RUN_DIR.name, "| step", ck.get("global_step"),
      "| image_encoder:", model.conditioner.image_encoder_kind,
      "| Dq =", model.conditioner.decoder.texel_dim)
print("params: conditioner %.2fM + dit %.2fM" % (
    sum(x.numel() for x in model.conditioner.parameters()) / 1e6,
    sum(x.numel() for x in model.dit.parameters()) / 1e6))
print("note: OA-80 prototype trains on all 80 objects — generalization "
      "is deliberately NOT this prototype's question")

## A. Input — stochastic rendered image / canonical mesh / GT / queries

In [ ]:
sid = SAMPLE_ID or man[int(rng.integers(len(man)))]["sample_id"]
it = TopoTexDataset(str(OA_ROOT / "dataset"), [sid], device=DEVICE)[0]
VIEW_K = int(rng.integers(6))          # which stochastic view conditions the model
imgs = (it["mv_images"].float() / 255)[None]
feed = imgs.clone(); feed[:, 0] = imgs[:, VIEW_K]
vmeta = json.loads((OA_ROOT / "oa100" / sid / "images/view_meta.json").read_text())
V3 = it["mesh"]["vertices"].cpu().numpy().astype(np.float64)
F3 = it["mesh"]["faces"].cpu().numpy().astype(np.int64)
ALLQ = {q["name"]: q for q in it["uv_queries"]}
fig, axes = plt.subplots(1, 6, figsize=(21, 3.4))
axes[0].imshow(imgs[0, VIEW_K].permute(1, 2, 0).cpu().numpy())
axes[0].set_title(f"input image = view_{VIEW_K:03d}\naz {vmeta[VIEW_K]['azimuth']:.0f} el {vmeta[VIEW_K]['elevation']:.0f} (record-only)", fontsize=8)
ctr = V3[F3].mean(1)
axes[1].scatter(ctr[:, 0], ctr[:, 1], s=1.0, c="k"); axes[1].set_aspect("equal")
axes[1].set_title(f"canonical mesh ({len(F3)} faces)", fontsize=8)
axes[2].imshow(it["uv_queries"][0]["gt_texture"].permute(1, 2, 0).cpu().numpy())
axes[2].set_title("native GT texture", fontsize=8)
for j, (key, nm) in enumerate([("uv_000", "native"), ("uv_001", "xatlas"), ("uv_002", "partial")]):
    axes[3 + j].imshow(ALLQ[key]["valid_mask"].cpu().numpy(), cmap="gray")
    axes[3 + j].set_title(f"{nm} valid mask", fontsize=8)
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.show()
print("object:", sid, "|", cat_of.get(sid), "| conditioning view:", VIEW_K)

## B+C+D. Image branch / Mesh branch / Fusion → Z_F

In [ ]:
cond = model.conditioner
with torch.no_grad():
    img_tok = cond.image_encoder(feed[:, 0])
    graph = it["graph"]
    pe = cond.topo_pe(graph, len(it["mesh"]["faces"]))
    face_tok = cond.tokenizer(it["mesh"]["vertices"], it["mesh"]["faces"], graph, pe)
    fused = cond.cross(face_tok.unsqueeze(0), img_tok).squeeze(0)
Z_F = pipe.encode(it["mesh"], feed[0], it["graph"])  # float input already in [0,1]
print("image tokens:", tuple(img_tok.shape), "| face tokens:", tuple(face_tok.shape),
      "| fused:", tuple(fused.shape), "| Z_F:", tuple(Z_F.shape))

from topotex.data.mesh import camera_matrices, rasterize_view
def face_render(cols, az=30, el=20, res=384):
    gb = rasterize_view(V3, F3, camera_matrices(az, el, V3.min(0), V3.max(0)), res)
    img = np.ones((res, res, 3)); img[gb["mask"]] = cols[gb["face_id"][gb["mask"]]]
    return img
def pca_rgb(x):
    x = np.asarray(x, np.float64); xc = x - x.mean(0)
    _, _, Vt = np.linalg.svd(xc[:: max(1, len(xc) // 4096)], full_matrices=False)
    pc = xc @ Vt[:3].T
    if pc.shape[1] < 3: pc = np.pad(pc, ((0, 0), (0, 3 - pc.shape[1])))
    lo, hi = np.percentile(pc, 2, 0), np.percentile(pc, 98, 0)
    return np.clip((pc - lo) / (hi - lo + 1e-9), 0, 1)

g16 = int(np.sqrt(img_tok.shape[1]))
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
axes[0].imshow(pca_rgb(img_tok[0].float().cpu().numpy()).reshape(g16, g16, 3), interpolation="nearest")
axes[0].set_title(f"B. image tokens PCA ({g16}x{g16}x384)", fontsize=9)
axes[1].imshow(face_render(pca_rgb(face_tok.float().cpu().numpy())))
axes[1].set_title("C. initial face tokens PCA", fontsize=9)
axes[2].imshow(face_render(pca_rgb(fused.float().cpu().numpy())))
axes[2].set_title("D. cross-attention output PCA", fontsize=9)
axes[3].imshow(face_render(pca_rgb(Z_F.float().cpu().numpy())))
axes[3].set_title("D'. Z_F PCA on mesh", fontsize=9)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## E. Surface query internals（factorized encoder，hook 捕获 + shape 断言）

In [ ]:
dec = model.conditioner.decoder
caps = {}
hooks = [
    dec.face_proj.register_forward_hook(lambda m, i, o: caps.__setitem__("addr", o.detach())),
    dec.bary_mlp.register_forward_hook(lambda m, i, o: caps.__setitem__("bary_v", o.detach())),
    dec.patch_embed.register_forward_pre_hook(lambda m, i: caps.__setitem__("dense", i[0].detach()[0])),
    dec.patch_embed.register_forward_hook(lambda m, i, o: caps.__setitem__("tok", o.detach()[0].flatten(1).T)),
    dec.norm.register_forward_hook(lambda m, i, o: caps.__setitem__("attn", o.detach())),
]
q0 = ALLQ["uv_000"]
with torch.no_grad():
    out = model.condition(Z_F, q0)
for h in hooks: h.remove()
H = W = dec.res
valid0 = q0["face_id"] >= 0
bmap = torch.zeros(H * W, dec.texel_dim, device=DEVICE)
bmap[valid0.reshape(-1)] = caps["bary_v"].float()
caps["bary_map"] = bmap.view(H, W, -1)
Fn = len(it["mesh"]["faces"])
shapes = {
    "face_address_table": (tuple(caps["addr"].shape), (Fn, 96)),
    "bary_address_map": (tuple(caps["bary_map"].shape), (H, W, 96)),
    "dense_texel_query": (tuple(caps["dense"].shape), (96, 256, 256)),
    "UV patch tokens": (tuple(caps["tok"].shape), (1024, 384)),
    "global_attention_out": (tuple(caps["attn"].shape), (1024, 384)),
    "uv_condition": (tuple(out["uv_condition"].shape[1:]), (64, 256, 256)),
}
for k, (got, want) in shapes.items():
    print(f"{'PASS' if got == want else 'FAIL'} {k:22s} {got}")
    assert got == want, k
vm_np = valid0.cpu().numpy()
panels = [(face_render(pca_rgb(caps["addr"].float().cpu().numpy())), "face address [F,96] PCA")]
img = pca_rgb(caps["bary_map"].cpu().numpy().reshape(-1, 96)).reshape(H, W, 3); img[~vm_np] = 1
panels.append((img, "bary address map PCA"))
img = pca_rgb(caps["dense"].float().cpu().numpy().reshape(96, -1).T).reshape(H, W, 3); img[~vm_np] = 1
panels.append((img, "dense texel query PCA"))
panels.append((pca_rgb(caps["tok"].float().cpu().numpy()).reshape(32, 32, 3), "UV patch tokens PCA"))
panels.append((pca_rgb(caps["attn"].float().cpu().numpy()).reshape(32, 32, 3), "attention output PCA"))
cx = out["uv_condition"][0].float().cpu().numpy()
img = pca_rgb(cx.reshape(64, -1).T).reshape(H, W, 3); img[~vm_np] = 1
panels.append((img, "uv_condition PCA"))
fig, axes = plt.subplots(1, 6, figsize=(21, 3.8))
for ax, (im, ttl) in zip(axes, panels):
    ax.imshow(im, interpolation="nearest" if im.shape[0] == 32 else None)
    ax.set_title(ttl, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## F. Generation — FM trajectory / 三查询 GT·预测·误差·渲染 / seam heatmap

In [ ]:
from topotex.data.mesh import (dilate_texture, linear_to_srgb_u8,
                               render_albedo_rebake, seam_error)

def render_tex(q, tex_u8, az=30, el=20, res=384):
    canon = types.SimpleNamespace(vertices=V3, faces=F3)
    uvr = types.SimpleNamespace(uv_vertices=q["uv_vertices"].astype(np.float64),
                                uv_faces=q["uv_faces"],
                                uv_face_to_mesh_face=np.arange(len(F3)))
    gb = rasterize_view(V3, F3, camera_matrices(az, el, V3.min(0), V3.max(0)), res)
    v = q["valid_mask"].cpu().numpy(); t = tex_u8.copy(); t[~v] = 0
    return linear_to_srgb_u8(render_albedo_rebake(canon, uvr, dilate_texture(t, v), gb)), gb["mask"]

def psnr(a, b, m):
    mse = float(((a[m] / 255. - b[m] / 255.) ** 2).mean())
    return round(10 * np.log10(1 / max(mse, 1e-12)), 2)

# FM trajectory (one Euler-50 run, fixed seed, native query)
m1 = q0["valid_mask"].float()[None, None]
gg = torch.Generator(device=DEVICE).manual_seed(SEED)
x = torch.randn(1, 3, H, W, device=DEVICE, generator=gg) * m1
taus = torch.linspace(1.0, 0.0, 51, device=DEVICE)
snaps = {1.0: x[0].clone()}
with torch.no_grad():
    for i in range(50):
        t_int = (taus[i] * model.schedule.T).round().clamp(min=1).long()
        v = model.dit(x, out["uv_condition"], m1, t_int.expand(1))
        x = (x - (taus[i] - taus[i + 1]) * v) * m1
        for tv in (0.75, 0.5, 0.25):
            if tv not in snaps and float(taus[i + 1]) <= tv + 1e-6:
                snaps[tv] = x[0].clone()
snaps[0.0] = x[0].clone()
vmq = q0["valid_mask"].cpu().numpy()
def to_im(tt):
    im = ((tt.clamp(-1, 1) + 1) / 2 * 255).round().byte().permute(1, 2, 0).cpu().numpy().copy()
    im[~vmq] = 0
    return im
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, tv in zip(axes, (1.0, 0.75, 0.5, 0.25, 0.0)):
    ax.imshow(to_im(snaps[tv])); ax.set_title(f"x @ tau={tv}", fontsize=9); ax.axis("off")
axes[5].imshow(q0["gt_texture"].permute(1, 2, 0).cpu().numpy())
axes[5].set_title("GT", fontsize=9); axes[5].axis("off")
plt.suptitle(f"Euler-50 trajectory — native query, seed {SEED}", fontsize=10)
plt.tight_layout(); plt.show()

preds, mets = {}, {}
fig, axes = plt.subplots(3, 5, figsize=(15, 9.4))
for r_i, (nm, key) in enumerate([("native", "uv_000"), ("xatlas", "uv_001"), ("partial", "uv_002")]):
    q = ALLQ[key]
    with torch.no_grad():
        o = model.condition(Z_F, q)
    xg = model.generate(o["uv_condition"], q["valid_mask"], num_steps=50, seed=SEED)
    im = ((xg.clamp(-1, 1) + 1) / 2 * 255).round().byte().permute(1, 2, 0).cpu().numpy().copy()
    vm = q["valid_mask"].cpu().numpy(); im[~vm] = 0
    preds[key] = im
    gt = (q["gt_texture"].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    mets[nm] = psnr(gt, im, vm)
    ra, _ = render_tex(q, im); rb, _ = render_tex(q, gt)
    axes[r_i][0].imshow(gt); axes[r_i][1].imshow(im)
    axes[r_i][2].imshow(np.abs(gt.astype(int) - im.astype(int)).sum(-1), cmap="inferno")
    axes[r_i][3].imshow(ra); axes[r_i][4].imshow(rb)
    axes[r_i][0].set_ylabel(f"{nm}\n{mets[nm]:.1f} dB", fontsize=10)
for c_i, t in enumerate(["GT", "generated", "|error|", "gen render", "GT render"]):
    axes[0][c_i].set_title(t, fontsize=10)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

q1 = ALLQ["uv_001"]
vm1 = q1["valid_mask"].cpu().numpy()
gt1 = (q1["gt_texture"].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
s_gen = seam_error(F3, q1["uv_vertices"], q1["uv_faces"], preds["uv_001"], vm1)
s_gt = seam_error(F3, q1["uv_vertices"], q1["uv_faces"], gt1, vm1)
err = s_gen["per_face_error"]
cm = plt.get_cmap("inferno")(np.clip(err / max(err.max(), 1e-6), 0, 1))[:, :3]
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(face_render(cm)); ax.set_title("seam heatmap (xatlas)", fontsize=9); ax.axis("off")
plt.show()
cons = []
for az, el in ((30, 20), (150, 20), (270, 20)):
    ia, ma = render_tex(ALLQ["uv_000"], preds["uv_000"], az, el)
    ib, mb = render_tex(q1, preds["uv_001"], az, el)
    mm = ma & mb
    if mm.sum() >= 100: cons.append(psnr(ia, ib, mm))
print("UV PSNR:", mets)
print(f"cross-layout consistency (native vs xatlas): {np.mean(cons):.2f} dB")
print(f"seam: generated {s_gen['seam_error']:.4f} | GT floor {s_gt['seam_error']:.4f} "
      f"| ratio {s_gen['seam_error'] / max(s_gt['seam_error'], 1e-9):.2f}")
pm = ALLQ["uv_002"]["valid_mask"].cpu().numpy()
print("partial outside-mask zero:", bool((preds["uv_002"][~pm] == 0).all()))

## G. View consistency — 同一 object 的 view_000–005 分别 forward

In [ ]:
zs_full = []
with torch.no_grad():
    for k in range(6):
        one = imgs.clone(); one[:, 0] = imgs[:, k]
        zk, _ = cond.encode_faces(it["mesh"], one, graph)
        zs_full.append(zk)
Zm = torch.stack([z_.mean(0) for z_ in zs_full]).float()
Zn = torch.nn.functional.normalize(Zm, dim=1)
cos = (Zn @ Zn.T).cpu().numpy()
dist = torch.cdist(Zm, Zm).cpu().numpy()
off = ~np.eye(6, dtype=bool)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
im0 = axes[0].imshow(cos, vmin=min(0.95, cos[off].min()), cmap="viridis"); plt.colorbar(im0, ax=axes[0])
axes[0].set_title(f"Z_F cosine — mean off-diag {cos[off].mean():.4f}", fontsize=9)
im1 = axes[1].imshow(dist, cmap="magma"); plt.colorbar(im1, ax=axes[1])
axes[1].set_title(f"Z_F L2 — mean off-diag {dist[off].mean():.3f}", fontsize=9)
for ax in axes:
    ax.set_xticks(range(6), [f"v{k}" for k in range(6)], fontsize=7)
    ax.set_yticks(range(6), [f"v{k}" for k in range(6)], fontsize=7)
plt.tight_layout(); plt.show()
allz = torch.cat(zs_full, 0).float().cpu().numpy()
xc = allz - allz.mean(0)
_, _, Vt6 = np.linalg.svd(xc[:: max(1, len(xc) // 4096)], full_matrices=False)
fig, axes = plt.subplots(1, 6, figsize=(21, 3.6))
for k in range(6):
    pc = (zs_full[k].float().cpu().numpy() - allz.mean(0)) @ Vt6[:3].T
    lo, hi = np.percentile(pc, 2, 0), np.percentile(pc, 98, 0)
    colv = np.clip((pc - lo) / (hi - lo + 1e-9), 0, 1)
    axes[k].imshow(face_render(colv)); axes[k].set_title(f"Z_F PCA | view {k}", fontsize=9); axes[k].axis("off")
plt.suptitle("shared-PCA face coloring — identical colors across views = view-invariant Z_F", fontsize=9)
plt.tight_layout(); plt.show()

## H. Image-condition probe — correct / shuffled / blank / noise

In [ ]:
donor = man[(next(i for i, m_ in enumerate(man) if m_["sample_id"] == sid) + 7) % len(man)]["sample_id"]
dit_ = TopoTexDataset(str(OA_ROOT / "dataset"), [donor], device=DEVICE)[0]
variants = {
    "correct": imgs[:, VIEW_K],
    "shuffled": (dit_["mv_images"].float() / 255)[None][:, 0],
    "blank": torch.full_like(imgs[:, 0], 235 / 255.0),
    "noise": torch.rand_like(imgs[:, 0]),
}
gt0 = (q0["gt_texture"].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
stats_h = {}
for j, (nm, img) in enumerate(variants.items()):
    fd = imgs.clone(); fd[:, 0] = img
    with torch.no_grad():
        z, _ = cond.encode_faces(it["mesh"], fd, graph)
        o = model.condition(z, q0)
    xg = model.generate(o["uv_condition"], q0["valid_mask"], num_steps=50, seed=SEED)
    tex = ((xg.clamp(-1, 1) + 1) / 2 * 255).round().byte().permute(1, 2, 0).cpu().numpy().copy()
    tex[~vmq] = 0
    rr, _ = render_tex(q0, tex)
    stats_h[nm] = psnr(gt0, tex, vmq)
    axes[0][j].imshow(img[0].permute(1, 2, 0).cpu().numpy()); axes[0][j].set_title(f"{nm} image", fontsize=9)
    axes[1][j].imshow(tex); axes[1][j].set_title(f"texture — {stats_h[nm]:.1f} dB vs GT", fontsize=9)
    axes[2][j].imshow(rr); axes[2][j].set_title("render", fontsize=9)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("PSNR vs GT:", stats_h)
print("expectation: correct > blank/noise, and shuffled changes appearance "
      "=> the model actually uses the image condition")